# Donaldson-Approach:

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


class AugmentedHybridModel(nn.Module):
    def __init__(self, p=1, q=1, s=10, d=1, m=2):
        super().__init__()
        self.p = p
        self.q = q
        self.s = s
        self.d = d
        self.m = m

        # GARCH parameters: omega, alpha_i, beta_j
        self.omega = nn.Parameter(torch.tensor(0.1))
        self.alpha = nn.Parameter(torch.tensor([0.1] * q))
        self.beta = nn.Parameter(torch.tensor([0.8] * p))

        # Zeta: coefficients for sigmoid basis functions
        self.zeta = nn.Parameter(torch.randn(s) * 0.01)

        # Lambda: fixed random weights for sigmoid arguments
        self.lambda_hdw = nn.Parameter(0.5*torch.empty((s, d, m)).uniform_(-1, 1), requires_grad=False)
        
        # Bias term per basis function
        self.bias = nn.Parameter(torch.zeros(s))  

    def forward(self, eps, sigma_sq):
        T = len(eps)
        vals = []

        eps_mean = eps.mean()
        eps_std = eps.std()
        eps_std = eps_std if eps_std > 1e-6 else torch.tensor(1.0)

        for t in range(T):
            garch_sum = torch.sum(self.alpha * eps[max(0, t - self.q):t] ** 2) if t > 0 else 0.0
            beta_sum = torch.sum(self.beta * sigma_sq[max(0, t - self.p):t]) if t > 0 else 0.0
            garch_part = self.omega + garch_sum + beta_sum

            # Compute z_t vector
            z_t = [(eps[t - dd] - eps_mean) / eps_std if (t - dd) >= 0 else 0.0 for dd in range(1, self.d + 1)]
            z_t = torch.tensor(z_t)

            # Compute sigmoid basis expansion
            basis = []
            for h_idx in range(self.s):
                lin_combo = 0.0
                for dd in range(self.d):
                    for ww in range(self.m):
                        lin_combo += self.lambda_hdw[h_idx, dd, ww] * z_t[dd]  # simplified: z^w = z
                activation = torch.sigmoid(lin_combo)
                basis.append(activation)
            basis = torch.stack(basis)
            correction = torch.sum(self.zeta * basis)

            val = garch_part + correction
            val = torch.clamp(val, min=1e-4)#, max=100.0)
            vals.append(val)

        h = torch.stack(vals)
        return h

# ---- Dataset Creation ----
def create_lagged_dataset(returns_series: pd.Series, lags: int = 1):
    returns = returns_series.values.flatten()
    T = len(returns)
    X_list = []
    y_list = []
    index_list = []

    for t in range(lags, T):
        lagged_input = returns[t - lags:t][::-1]  
        target = returns[t] ** 2  
        X_list.append(lagged_input)
        y_list.append(target)
        index_list.append(returns_series.index[t])

    X = torch.tensor(np.array(X_list)).float()
    y = torch.tensor(np.array(y_list)).float()
    return X, y, pd.Index(index_list)


def negative_log_likelihood(y_true, y_pred):
    y_pred = torch.clamp(y_pred, min=1e-6)
    return 0.5 * torch.mean(torch.log(y_pred) + y_true / y_pred)

def train_augmented_model(returns_series, lags=1, epochs=100):
    X, y, _ = create_lagged_dataset(returns_series, lags=lags)
    model = AugmentedHybridModel()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    losses = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        h = model(X[:, 0], X[:, 0] ** 2)  
        loss = negative_log_likelihood(y, h)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.6f}")

    return model, losses

def evaluate_reference_model(model, test_series,ModelName, RealizedVolatility = None):
    test_tensor = torch.tensor(test_series.values.flatten()).float()
    with torch.no_grad():
        h = model(test_tensor, test_tensor ** 2)
        vol = torch.sqrt(h).detach().numpy()
    squared_returns = test_tensor.numpy() ** 2
    mse = np.mean((squared_returns - h.numpy()) ** 2)
    print(f"MSE between squared returns and predicted variance: {mse:.6f}")
    plt.figure(figsize=(12, 5))
    plt.plot(test_series.index, h.numpy(), label="Predicted Variance", linewidth=2,linestyle='dashed')
    plt.plot(test_series.index, squared_returns, alpha=0.4, label="Squared Returns")
    plt.title(f"{ModelName}: Predicted Volatility vs. Squared Returns")
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    return(h)

def run_reference_model_with_split(returns_series, train_ratio=0.8, lags=1, epochs=100):
    split_idx = int(len(returns_series) * train_ratio)
    train_series = returns_series.iloc[:split_idx]
    test_series = returns_series.iloc[split_idx:]

    model, losses = train_reference_model(train_series, lags=lags, epochs=epochs)
    evaluate_reference_model(model, test_series)
    return model, losses


# Extended-Approach: MLP

In [ ]:
import torch.nn.functional as F
class AugmentedModel_Extended_mlp(nn.Module):
    def __init__(self, input_size=2, hidden_size=10):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.tanh(x)
        return x

class ConstrainedGARCHParams(nn.Module):
    def __init__(self, init_omega=0.1, init_alpha=0.1, init_beta=0.8):
        super().__init__()
        self.raw_omega = nn.Parameter(torch.tensor([init_omega]).log())
        self.raw_alpha = nn.Parameter(torch.tensor(init_alpha).logit())
        self.raw_beta = nn.Parameter(torch.tensor(init_beta).logit())

    def forward(self):
        omega = F.softplus(self.raw_omega)
        alpha = torch.sigmoid(self.raw_alpha)
        beta = torch.sigmoid(self.raw_beta) * (1 - alpha)
        return omega.squeeze(), alpha.squeeze(), beta.squeeze()

def compute_volatility_joint(returns_tensor, net, garch_param_module):
    T = len(returns_tensor)
    h_list = []
    h_prev = torch.var(returns_tensor)

    omega, alpha, beta = garch_param_module()

    for t in range(T):
        if t == 0:
            h_t = h_prev
        else:
            r2 = returns_tensor[t - 1] ** 2
            h_detached = h_prev.detach()
            
            # New
            mean_r2 = returns_tensor.pow(2).mean().item()
            mean_h = torch.var(returns_tensor).item()
            nn_input = torch.stack([r2 / mean_r2, h_detached / mean_h]).unsqueeze(0)
            # Old
            #nn_input = torch.stack([r2, h_detached]).unsqueeze(0)
            nn_adjust = net(nn_input).squeeze() #* 0.5
            h_t = omega + alpha * r2 + beta * h_prev + nn_adjust
            h_t = torch.clamp(h_t, min=1e-4)#, max=5.0)

        h_list.append(h_t.view(()))
        h_prev = h_t

    return torch.stack(h_list),nn_adjust


import torch

@torch.no_grad()  
def compute_volatility_new(
    returns_tensor: torch.Tensor,   
    net: torch.nn.Module,             
    garch_param_module,               
    r2_mean: torch.Tensor=None,
    r2_std: torch.Tensor=None,
    h_mean: torch.Tensor=None,
    h_std: torch.Tensor=None,
    min_var: float = 1e-4
):
    
    device = returns_tensor.device
    dtype  = returns_tensor.dtype
    T = returns_tensor.shape[0]

    r2_series = returns_tensor.pow(2)

    
    if r2_mean is None: r2_mean = r2_series.mean()
    if r2_std  is None: r2_std  = r2_series.std(unbiased=False)
    if h_mean  is None: h_mean  = returns_tensor.var(unbiased=False)  
    if h_std   is None: h_std   = torch.sqrt(h_mean)

    tiny = torch.tensor(1e-8, device=device, dtype=dtype)
    r2_std = torch.clamp(r2_std, min=tiny)
    h_std  = torch.clamp(h_std,  min=tiny)

    
    h_prev = returns_tensor.var(unbiased=False)

    omega, alpha, beta = garch_param_module()  # tensors on correct device/dtype

    h_list = []
    nn_adjust_last = torch.zeros((), device=device, dtype=dtype)

    for t in range(T):
        if t == 0:
            h_t = h_prev
        else:
            r2 = r2_series[t - 1]
            h_detached = h_prev.detach()

            x1 = (r2 - r2_mean) / r2_std
            x2 = (h_detached - h_mean) / h_std
            nn_input = torch.stack([x1, x2]).unsqueeze(0)  

            nn_adjust = net(nn_input).squeeze()
            nn_adjust_last = nn_adjust

            h_t = omega + alpha * r2 + beta * h_prev + nn_adjust
            h_t = torch.clamp(h_t, min=torch.tensor(min_var, device=device, dtype=dtype))

        h_list.append(h_t.view(()))
        h_prev = h_t

    return torch.stack(h_list), nn_adjust_last


def negative_log_likelihood(returns_tensor, h):
    h = torch.clamp(h, min=1e-6)
    eps = returns_tensor / torch.sqrt(h)
    nll = 0.5 * (torch.log(h) + eps ** 2)
    return nll.mean()

def run_joint_training(aug_model, returns_series, epochs=100, init_omega=0.1, init_alpha=0.1, init_beta=0.8):
    returns_tensor = torch.tensor(returns_series.values.flatten()).float()

    net = aug_model
    learned_garch_params = ConstrainedGARCHParams(init_omega=init_omega, init_alpha=init_alpha, init_beta=init_beta)

    params = list(net.parameters()) + list(learned_garch_params.parameters())
    optimizer = optim.Adam(params, lr=0.001)

    loss_history = []

    for epoch in range(epochs):
        net.train()
        optimizer.zero_grad()

        h,nn_adjust = compute_volatility_joint(returns_tensor, net, learned_garch_params)
        loss = negative_log_likelihood(returns_tensor, h)

        if torch.isnan(loss):
            print("NaN detected in loss!")
            break

        loss.backward()
        optimizer.step()
        
        weight_means = {
            name: np.round(param.data.mean().item(),4)
            for name, param in net.named_parameters()
            if 'weight' in name
        }

        loss_history.append(loss.item())
        omega, alpha, beta = learned_garch_params()
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.6f} | ω={omega.item():.4f}, α={alpha.item():.4f}, β={beta.item():.4f}, α+β={alpha.item() + beta.item():.4f} | NN mean weights: {weight_means} | NN adjust: {nn_adjust.item():.8f}")

    return net, loss_history, [omega,alpha,beta,nn_adjust] , learned_garch_params

In [ ]:
import numpy as np
import pandas as pd
import torch

@torch.no_grad()
def lower_triangular_from_workflow(
    net: torch.nn.Module,
    model_name,
    garch_param_module: torch.nn.Module,
    test_returns: pd.Series,
    H: int = 300,
    min_var: float = 1e-6,
    lower_triangluar = False
) -> pd.DataFrame:
   
    device = next(net.parameters()).device
    net.eval()


    r_full = torch.tensor(test_returns.values, dtype=torch.float32, device=device).view(-1, 1)
    N = r_full.shape[0]
    H = min(H, N)


    mean_r2 = (r_full.pow(2).mean()).item()
    mean_h  = (r_full.var(unbiased=True)).item()
    if not np.isfinite(mean_r2) or mean_r2 <= 0: mean_r2 = 1e-6
    if not np.isfinite(mean_h)  or mean_h  <= 0: mean_h  = mean_r2


    omega, alpha, beta = garch_param_module()
    omega, alpha, beta = (x.to(device).view(1, 1) for x in (omega, alpha, beta))

    h0_list = []
    h_prev = r_full.var(unbiased=True)  
    for t in range(N):
        if t == 0:
            h_t = h_prev
        else:
            r2_tm1 = (r_full[t - 1, 0] ** 2)      
            hprev  = h_prev.squeeze()            
            nn_in  = torch.stack([r2_tm1/mean_r2, hprev/mean_h]).view(1, 2)  # (1,2)

            nn_adj = net(nn_in).view(1, 1)
            h_t = omega + alpha * r2_tm1 + beta * h_prev + nn_adj
            h_t = torch.clamp(h_t, min=min_var)
        h0_list.append(h_t.view(1, 1))
        h_prev = h_t
    h_start = torch.cat(h0_list, dim=0)  


    mat = np.full((N, H), np.nan, dtype=np.float64)

    r2_curr = r_full.pow(2)      
    h_curr  = h_start.clone()   

    for h in range(1, H + 1):
        if lower_triangluar == True:
            count = N - h
            
        else:
            count = N
        if count <= 0:
            break

        feats  = torch.cat([r2_curr[:count] / mean_r2, h_curr[:count].detach() / mean_h], dim=1)  # (count,2)
        nn_adj = net(feats).view(-1, 1)
        v_hat  = omega + alpha * r2_curr[:count] + beta * h_curr[:count] + nn_adj
        v_hat  = torch.clamp(v_hat, min=min_var)

        mat[:count, h - 1] = v_hat.squeeze(-1).detach().cpu().numpy()

        r2_curr = v_hat
        h_curr  = v_hat

    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    n_step_preds_matrix = pd.DataFrame(mat, index=test_returns.index, columns=cols)
    
    n_step_qlikes = []
    for i in range(H):
        df_h_step = pd.concat([n_step_preds_matrix.iloc[:,i].dropna(),test_returns**2],axis=1).dropna()

        n_step_qlikes.append(forecast_metrics(preds = df_h_step.iloc[:,0],
                         y_test = df_h_step.iloc[:,1],
                         model = model_name)[-1])
    df_qlikes = pd.DataFrame(n_step_qlikes)
    df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv",header = False, index = False)
    return(df_qlikes,n_step_preds_matrix)

# Extended-Approach: LSTM

In [1]:
class AugmentedModel_Extended_LSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=10, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = x.unsqueeze(1)  
        lstm_out, _ = self.lstm(x)  
        out = self.fc(lstm_out[:, -1, :])
        return self.tanh(out)

NameError: name 'nn' is not defined

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from sklearn.preprocessing import StandardScaler

class ConstrainedGARCHParamsTF(tf.keras.Model):
    def __init__(self, init_omega=0.1, init_alpha=0.1, init_beta=0.8):
        super().__init__()
        # raw trainable variables
        self.raw_omega = tf.Variable(tf.math.log(tf.constant([init_omega], dtype=tf.float32)), trainable=True)
        self.raw_alpha = tf.Variable(tf.math.log(init_alpha / (1 - init_alpha)), trainable=True)
        self.raw_beta  = tf.Variable(tf.math.log(init_beta / (1 - init_beta)), trainable=True)

    def __call__(self):
        omega = tf.nn.softplus(self.raw_omega)
        alpha = tf.nn.sigmoid(self.raw_alpha)
        beta  = tf.nn.sigmoid(self.raw_beta) * (1 - alpha)
        return tf.squeeze(omega), tf.squeeze(alpha), tf.squeeze(beta)


def build_lstm_adjustment_model(input_dim=2, lstm_units=16): 
    inputs = Input(shape=(1, input_dim)) 
    x = LSTM(lstm_units, return_sequences=False)(inputs)
    x = Dense(8, activation='relu')(x)
    output = Dense(1, activation='tanh')(x)
    return tf.keras.Model(inputs=inputs, outputs=output)

def compute_volatility_lstm_old(returns_tensor, net, garch_params):

    T = len(returns_tensor)
    h_list = []
    h_prev = tf.math.reduce_variance(returns_tensor)
    nn_adjust_last = tf.constant(0.0)  # Default

    omega, alpha, beta = garch_params()

    for t in range(T):
        if t == 0:
            h_t = h_prev
        else:
            r2 = tf.square(returns_tensor[t - 1])
            h_detached = tf.stop_gradient(h_prev)

            features = tf.stack([r2, h_detached], axis=0)
            features = tf.expand_dims(features, axis=0)
            nn_input = tf.expand_dims(features, axis=1)

            nn_adjust = net(nn_input)[0][0]
            nn_adjust_last = nn_adjust 

            h_t = omega + alpha * r2 + beta * h_prev + nn_adjust
            h_t = tf.clip_by_value(h_t, clip_value_min=1e-4, clip_value_max=1e4)

        h_list.append(tf.reshape(h_t, ()))
        h_prev = h_t

    return tf.stack(h_list), nn_adjust_last

def compute_volatility_lstm_before(returns_tensor, net, garch_params):
    T = len(returns_tensor)
    h_list = []
    h_prev = tf.math.reduce_variance(returns_tensor)
    nn_adjust_last = tf.constant(0.0)

    r2_mean = tf.reduce_mean(tf.square(returns_tensor))
    h_mean = tf.math.reduce_variance(returns_tensor)

    omega, alpha, beta = garch_params()

    for t in range(T):
        if t == 0:
            h_t = h_prev
        else:
            r2 = tf.square(returns_tensor[t - 1])
            h_detached = tf.stop_gradient(h_prev)

            r2_norm = r2 / r2_mean
            h_norm = h_detached / h_mean
            features = tf.stack([r2_norm, h_norm], axis=0)

            features = tf.expand_dims(features, axis=0)  
            nn_input = tf.expand_dims(features, axis=1)  

            nn_adjust = net(nn_input)[0][0]
            nn_adjust_last = nn_adjust

            h_t = omega + alpha * r2 + beta * h_prev + nn_adjust
            h_t = tf.clip_by_value(h_t, clip_value_min=1e-4, clip_value_max=1e4)

        h_list.append(tf.reshape(h_t, ()))
        h_prev = h_t

    return tf.stack(h_list), nn_adjust_last

import tensorflow as tf

def compute_volatility_lstm(
    returns_tensor,    
    net,               
    garch_params,       
    r2_mean=None,
    r2_std=None,
    h_mean=None,
    h_std=None,
    min_var=1e-4,
    max_var=1e4
):

    returns_tensor = tf.convert_to_tensor(returns_tensor)
    T = tf.shape(returns_tensor)[0]

    r2_series = tf.square(returns_tensor)

    # --- Stats ---
    if r2_mean is None:
        r2_mean = tf.reduce_mean(r2_series)
    if r2_std is None:
        r2_std = tf.math.reduce_std(r2_series)  
    if h_mean is None:
        h_mean = tf.math.reduce_variance(returns_tensor)  
    if h_std is None:
        h_std = tf.sqrt(h_mean)

    tiny = tf.constant(1e-8, dtype=returns_tensor.dtype)
    r2_std = tf.maximum(r2_std, tiny)
    h_std  = tf.maximum(h_std,  tiny)

    h_prev = tf.math.reduce_variance(returns_tensor)
    nn_adjust_last = tf.constant(0.0, dtype=returns_tensor.dtype)

    omega, alpha, beta = garch_params()

    h_list = []
    for t in tf.range(T):
        def step_first():
            return h_prev, nn_adjust_last

        def step_body():
            r2 = r2_series[t - 1]
            h_detached = tf.stop_gradient(h_prev)

            x1 = (r2 - r2_mean) / r2_std
            x2 = (h_detached - h_mean) / h_std
            feats = tf.stack([x1, x2], axis=0)      # (2,)
            feats = tf.expand_dims(feats, 0)        # (1, 2)
            nn_input = tf.expand_dims(feats, 1)     # (1, 1, 2)

            nn_adjust = net(nn_input)[:, 0]         # (1,) -> (1,)
            nn_adjust_scalar = nn_adjust[0]

            h_t = omega + alpha * r2 + beta * h_prev + nn_adjust_scalar
            h_t = tf.clip_by_value(h_t, min_var, max_var)
            return h_t, nn_adjust_scalar

        h_t, nn_adjust_last = tf.cond(tf.equal(t, 0), step_first, step_body)

        h_list.append(tf.reshape(h_t, ()))
        h_prev = h_t

    return tf.stack(h_list), nn_adjust_last


import tensorflow as tf

@tf.function() 
def compute_volatility_lstm_fast(
    returns_tensor,
    net,
    garch_params,
    r2_mean=None, r2_std=None,
    h_mean=None,  h_std=None,
    min_var=1e-4, max_var=1e4
):
    x = tf.convert_to_tensor(returns_tensor)
    x = tf.cast(x, tf.float32)                
    T = tf.shape(x)[0]
    r2_series = tf.square(x)

    
    r2_mean = tf.reduce_mean(r2_series) if r2_mean is None else tf.cast(r2_mean, x.dtype)
    r2_std  = tf.math.reduce_std(r2_series) if r2_std  is None else tf.cast(r2_std,  x.dtype)
    h_var   = tf.math.reduce_variance(x)      # compute once
    h_mean  = h_var if h_mean is None else tf.cast(h_mean, x.dtype)
    h_std   = tf.sqrt(h_mean) if h_std is None else tf.cast(h_std, x.dtype)

    tiny = tf.constant(1e-8, x.dtype)
    r2_std = tf.maximum(r2_std, tiny)
    h_std  = tf.maximum(h_std,  tiny)

    omega, alpha, beta = garch_params()
    omega = tf.cast(omega, x.dtype)
    alpha = tf.cast(alpha, x.dtype)
    beta  = tf.cast(beta,  x.dtype)

    
    h_ta = tf.TensorArray(x.dtype, size=T)

    
    h_prev = h_var
    nn_adjust_last = tf.constant(0.0, x.dtype)
    h_ta = h_ta.write(0, tf.clip_by_value(h_prev, min_var, max_var))

   
    def body(t, h_prev, nn_last, h_ta):
        r2 = r2_series[t - 1]
        h_detached = tf.stop_gradient(h_prev)

        x1 = (r2 - r2_mean) / r2_std
        x2 = (h_detached - h_mean) / h_std
        nn_input = tf.reshape(tf.stack([x1, x2]), [1, 1, 2])   # (batch=1, time=1, feat=2)

        nn_adjust = net(nn_input)[:, 0]                        # (1,)
        nn_scalar = nn_adjust[0]

        h_t = omega + alpha * r2 + beta * h_prev + nn_scalar
        h_t = tf.clip_by_value(h_t, min_var, max_var)

        h_ta = h_ta.write(t, h_t)
        return t + 1, h_t, nn_scalar, h_ta

    
    cond = lambda t, *_: t < T
    _, h_last, nn_adjust_last, h_ta = tf.while_loop(
        cond, body, (tf.constant(1), h_prev, nn_adjust_last, h_ta),
        parallel_iterations=1,  # recurrence -> 1
    )

    return h_ta.stack(), nn_adjust_last


def negative_log_likelihood_tf(returns_tensor, h):
    h = tf.clip_by_value(h, 1e-6, 1e4)
    eps = returns_tensor / tf.sqrt(h)
    nll = 0.5 * (tf.math.log(h) + tf.square(eps))
    return tf.reduce_mean(nll)

def run_lstm_joint_training(returns_series, epochs=100, init_omega=0.1, init_alpha=0.1, init_beta=0.8):
    returns_tensor = tf.convert_to_tensor(returns_series.values.flatten(), dtype=tf.float32)

    net = build_lstm_adjustment_model()
    garch_params = ConstrainedGARCHParamsTF(init_omega=init_omega, init_alpha=init_alpha, init_beta=init_beta)
    garch_params.build(input_shape=())

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
    loss_history = []

    for epoch in range(epochs):
        with tf.GradientTape() as tape:
            h, nn_adjust_val = compute_volatility_lstm_fast(returns_tensor, net, garch_params)
            loss = negative_log_likelihood_tf(returns_tensor, h)

        grads = tape.gradient(loss, net.trainable_variables + garch_params.trainable_variables)
        optimizer.apply_gradients(zip(grads, net.trainable_variables + garch_params.trainable_variables))

        # Call AFTER GradientTape
        omega, alpha, beta = garch_params()
        omega_val = omega.numpy()
        alpha_val = alpha.numpy()
        beta_val = beta.numpy()
        
        loss_val = loss.numpy()

        print(f"Epoch {epoch:03d} | Loss: {loss_val:.6f} | ω={omega_val:.4f}, α={alpha_val:.4f}, β={beta_val:.4f}, α+β={alpha_val + beta_val:.4f} | nn_adjust={nn_adjust_val.numpy():.4f}")
        loss_history.append(loss_val)

    return net, garch_params, loss_history, nn_adjust_val


2025-07-12 14:56:21.691811: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

def build_lower_triangular_from_workflow_tf(
    net: tf.keras.Model,
    model_name,
    garch_params,                 
    test_returns: pd.Series,
    H: int = 300,
    min_var: float = 1e-6,
    lower_triangular = False
) -> pd.DataFrame:
    
    r_full = tf.convert_to_tensor(test_returns.values.astype("float32")) 
    N = int(r_full.shape[0])
    H = min(H, N)

    mean_r2 = tf.reduce_mean(tf.square(r_full))
    mean_h  = tf.math.reduce_variance(r_full)
    mean_r2 = tf.maximum(mean_r2, tf.constant(1e-6, dtype=mean_r2.dtype))
    mean_h  = tf.maximum(mean_h,  tf.constant(1e-6, dtype=mean_h.dtype))

    omega, alpha, beta = garch_params()
    omega = tf.cast(omega, tf.float32)
    alpha = tf.cast(alpha, tf.float32)
    beta  = tf.cast(beta,  tf.float32)

    h_list = []
    h_prev = tf.math.reduce_variance(r_full)

    for t in range(N):
        if t == 0:
            h_t = h_prev
        else:
            r2_tm1 = tf.square(r_full[t-1])
            h_det  = tf.stop_gradient(h_prev)

            feats = tf.stack([r2_tm1/mean_r2, h_det/mean_h], axis=0)  
            feats = tf.expand_dims(tf.expand_dims(feats, axis=0), axis=1)  

            nn_adj = net(feats)[0, 0]  # scalar
            h_t = omega + alpha * r2_tm1 + beta * h_prev + nn_adj
            h_t = tf.clip_by_value(h_t, min_var, 1e12)

        h_list.append(tf.reshape(h_t, (1, 1)))
        h_prev = h_t

    h_start = tf.concat(h_list, axis=0)

    mat = np.full((N, H), np.nan, dtype=np.float64)

    r2_curr = tf.square(tf.reshape(r_full, (-1, 1))) 
    h_curr  = tf.identity(h_start)

    for h in range(1, H + 1):
        if lower_triangular == True:
            count = N - h
        
        else:
            count = N
        if count <= 0:
            break

        r2_slice = r2_curr[:count] / mean_r2
        h_slice  = tf.stop_gradient(h_curr[:count]) / mean_h
        feats    = tf.concat([r2_slice, h_slice], axis=1)      
        feats    = tf.expand_dims(feats, axis=1)              

        nn_adj = net(feats)                                     
        nn_adj = tf.reshape(nn_adj, (-1, 1))                    

        v_hat = omega + alpha * r2_curr[:count] + beta * h_curr[:count] + nn_adj
        v_hat = tf.clip_by_value(v_hat, min_var, 1e12)

        mat[:count, h - 1] = v_hat.numpy().reshape(-1)

        r2_curr = tf.concat([v_hat, r2_curr[count:]], axis=0) 
        h_curr  = tf.concat([v_hat, h_curr[count:]], axis=0)
        
        print("Forecastin Horizon:",h)

    cols = [f"h.{k:03d}" for k in range(1, H + 1)]
    
    n_step_preds_matrix = pd.DataFrame(mat, index=test_returns.index, columns=cols)
    n_step_qlikes = []
    for i in range(H):#299):
        df_h_step = pd.concat([n_step_preds_matrix.iloc[:,i].dropna(),test_returns**2],axis=1).dropna()

        n_step_qlikes.append(forecast_metrics(preds = df_h_step.iloc[:,0],
                         y_test = df_h_step.iloc[:,1],
                         model = model_name)[-1])
    df_qlikes = pd.DataFrame(n_step_qlikes)
    #df_qlikes.to_csv(path + f"/Plots/Loss/{model_name}_loss_g{gamma}.csv",header = False, index = False)
    return(df_qlikes,n_step_preds_matrix)